# CS604 Project - 03 Make a submission zip from a finished run

For a run that has finished training: makes its CodaBench zip, and can re-score it on the hold-out with and without mirror averaging. **Nothing is trained here**, so Run All is safe.

Before you start: the run folder `My Drive/cs604_runs/<RUN>/` in YOUR Google Drive must hold that run's `config.yaml` and `final.pt` (for a run trained by someone else, copy those two files from their Drive).

In [ ]:
REPO_URL = "https://github.com/notzdean/CS604-Computer-Vision.git"   # <- set once
RUN    = ""             # name of the run folder inside My Drive/cs604_runs, for example "abl_adaptformer_only"
TEAM   = "XX"           # team number for the file name (val_team_XX.zip)
TTA    = "none"         # "none" = one pass per image. "flip" = mirror averaging (2 passes: report twice the latency)


## 1. Code

In [ ]:
import getpass, os, subprocess
REPO = "/content/repo"

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)
    return r.returncode, (r.stdout + r.stderr).strip()

def with_token(url, token):                      # the token is only used for this command, never stored in the repo
    return url.replace("https://", f"https://x-access-token:{token}@")

token = os.environ.get("GITHUB_TOKEN", "")
if not os.path.isdir(f"{REPO}/.git"):
    code, out = git("clone", REPO_URL, REPO)     # works if the repo is public
    if code != 0:
        print("Clone failed (private repo?). Paste a GitHub token with read access to this repo.")
        token = getpass.getpass("GitHub token: ")
        code, out = git("clone", with_token(REPO_URL, token), REPO)
        if code == 0:
            git("remote", "set-url", "origin", REPO_URL, cwd=REPO)   # do not keep the token in .git/config
    assert code == 0, out.replace(token, "***") if token else out
else:
    code, out = git("pull", with_token(REPO_URL, token) if token else "origin", cwd=REPO)
    print(out.replace(token, "***") if token else out)
print("repo ready at", REPO)
!pip install -q -r /content/repo/requirements.txt
# Colab's old torchao (0.10) breaks peft's LoRA import; we do not use torchao, so remove it
!pip uninstall -y -q torchao

## 2. Data (skipped if already there; the val images are needed for the predictions)

In [ ]:
%%bash
set -e
set -o pipefail
D=/content/cs604-course-data
U=https://huggingface.co/datasets/doem1997/cs604-course-data/resolve/main
if [ -f $D/.ready ]; then echo "Data already prepared - skipping download."; exit 0; fi
mkdir -p $D && cd $D
curl -sSfL $U/train.txt -o train.txt
curl -sSfL $U/val.txt -o val.txt
# stream both archives in parallel, extracting as they download
curl -sSfL $U/val.tar.gz   | tar -xz &
P1=$!
curl -sSfL $U/train.tar.gz | tar -xz
wait $P1
touch .ready
df -h /content
ls $D

## 3. Google Drive and the run folder

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
OUT = "/content/drive/MyDrive/cs604_runs"
run_dir = f"{OUT}/{RUN}"
assert RUN, "set RUN in the first cell"
missing = [f for f in ("config.yaml", "final.pt") if not os.path.exists(f"{run_dir}/{f}")]
assert not missing, f"not found in {run_dir}: {missing}. Upload them to My Drive/cs604_runs/{RUN}/ (a new upload can take a minute to appear), then run this cell again."
%cd /content/repo
print("using", run_dir, "->", sorted(os.listdir(run_dir)))

## 4. Optional check: hold-out score with and without mirror averaging
Takes a minute or two. Use it to decide whether `TTA = "flip"` is worth the doubled latency. It stops with a message for runs trained with `holdout_per_class=0` (they have no hold-out).

In [ ]:
!python -m common.evaluate $OUT/$RUN

## 5. Predict the val images and write the zip
Writes `val_team_XX.txt`, `val_team_XX.zip` and `val_top5.npz` into the run folder (replacing older ones there). Upload the zip to CodaBench.

In [ ]:
!python -m common.submit $OUT/$RUN --team $TEAM --split val --tta $TTA